# Notebook 04: Cardiomegaly Baseline — ResNet18

Same 3-stage progression, with **ResNet18** (pretrained on ImageNet). Lightweight classic residual architecture — ~11M params, comparable footprint to MobileNetV2 (3.5M) and EfficientNet-B0 (5.3M), but architecturally different (plain residual blocks, no depthwise separable conv, no squeeze-and-excitation) so its errors tend to decorrelate for ensembling.

- **Feature size**: 512 channels (replaces `model.fc`)
- **Fine-tune block**: `layer4` (the final residual block group)

---
## Step 1: Imports

In [ ]:
import copy
import random
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as T
from PIL import Image
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from torchvision.models import ResNet18_Weights, resnet18
from tqdm.auto import tqdm

print("Imports OK")

## Step 2: Reproducibility and Device

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print(f"Device: {device}")

## Step 3: Configuration

ResNet18 is small (~11M params), so you can push `IMG_SIZE` higher if you want — 384 or 512 should still be fast.

In [ ]:
REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
TRAIN_CSV = REPO / "data" / "train_val" / "train_val.csv"
TRAIN_IMG_DIR = REPO / "data" / "train_val" / "images"
TEST_IMG_DIR = REPO / "data" / "test_images"
PRED_DIR = REPO / "outputs" / "predictions"
CKPT_DIR = REPO / "outputs" / "checkpoints"
PRED_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR.mkdir(parents=True, exist_ok=True)

IMG_SIZE = 256          # ResNet50 is heavier; 256 is a good starting point. Try 384/512 later.
BATCH_SIZE = 16
NUM_WORKERS = 0
VAL_FRAC = 0.2

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

print(f"Train CSV : {TRAIN_CSV}")
print(f"Train imgs: {TRAIN_IMG_DIR}")
print(f"Test imgs : {TEST_IMG_DIR}")
print(f"IMG_SIZE  : {IMG_SIZE or 'native (1024)'}")

## Step 4: Load labels and inspect class balance

In [ ]:
df = pd.read_csv(TRAIN_CSV)
df = df.rename(columns={
    "Image Index": "image_file",
    "Patient Age": "age",
    "Patient Sex": "sex",
    "Finding Labels": "finding",
})
df["label"] = (df["finding"] == "Cardiomegaly").astype(int)

print(df.head())
print("\nFinding counts:")
print(df["finding"].value_counts())
print(f"\nPositive rate: {df['label'].mean():.3f}")
print(f"Total         : {len(df)}")

## Step 5: Dataset class

In [ ]:
class CardiomegalyDataset(Dataset):
    def __init__(self, df, img_dir, transform=None, return_label=True):
        self.df = df.reset_index(drop=True)
        self.img_dir = Path(img_dir)
        self.transform = transform
        self.return_label = return_label

    def __len__(self):
        return len(self.df)

    def _load_image(self, fname):
        img = Image.open(self.img_dir / fname)
        if img.mode != "RGB":
            img = img.convert("RGB")
        return img

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = self._load_image(row["image_file"])
        if self.transform is not None:
            img = self.transform(img)
        if not self.return_label:
            return img, row["image_file"]
        return img, torch.tensor(row["label"], dtype=torch.float32)

## Step 6: Stratified train / validation split

80/20, stratified on the label. `SEED=42` matches every other notebook, so the same images end up in train vs. val across 01, 02, 03, 04.

In [ ]:
train_df, val_df = train_test_split(
    df, test_size=VAL_FRAC, stratify=df["label"], random_state=SEED
)
print(f"Train: {len(train_df)}  (pos rate {train_df['label'].mean():.3f})")
print(f"Val  : {len(val_df)}  (pos rate {val_df['label'].mean():.3f})")

## Step 7: Transforms

In [ ]:
def build_transform(img_size, augment=False):
    ops = []
    if img_size is not None:
        ops.append(T.Resize((img_size, img_size)))
    if augment:
        ops += [
            T.RandomHorizontalFlip(p=0.5),
            T.RandomRotation(degrees=5),
            T.ColorJitter(brightness=0.1, contrast=0.1),
        ]
    ops += [T.ToTensor(), T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)]
    return T.Compose(ops)

train_tf = build_transform(IMG_SIZE, augment=False)
val_tf = build_transform(IMG_SIZE, augment=False)

train_ds = CardiomegalyDataset(train_df, TRAIN_IMG_DIR, transform=train_tf)
val_ds = CardiomegalyDataset(val_df, TRAIN_IMG_DIR, transform=val_tf)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

print(f"Train batches: {len(train_loader)}")
print(f"Val   batches: {len(val_loader)}")

---
## Metrics helpers

In [ ]:
def sens_spec(y_true, y_prob, threshold=0.5):
    y_true = np.asarray(y_true); y_prob = np.asarray(y_prob)
    pred = (y_prob >= threshold).astype(int)
    tp = int(((pred == 1) & (y_true == 1)).sum())
    tn = int(((pred == 0) & (y_true == 0)).sum())
    fp = int(((pred == 1) & (y_true == 0)).sum())
    fn = int(((pred == 0) & (y_true == 1)).sum())
    sens = tp / (tp + fn) if (tp + fn) else 0.0
    spec = tn / (tn + fp) if (tn + fp) else 0.0
    return sens, spec

def datathon_score(auroc, sens, spec):
    return 0.5 * auroc + 0.25 * sens + 0.25 * spec

def best_threshold(y_true, y_prob):
    fpr, tpr, thr = roc_curve(y_true, y_prob)
    return float(thr[np.argmax(tpr - fpr)])

## Train / Eval loops

In [ ]:
@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    ys, ps = [], []
    for imgs, labels in loader:
        imgs = imgs.to(device)
        logits = model(imgs).squeeze(1)
        probs = torch.sigmoid(logits).detach().cpu().numpy()
        ys.extend(labels.numpy().tolist())
        ps.extend(probs.tolist())
    ys = np.array(ys); ps = np.array(ps)
    auroc = roc_auc_score(ys, ps)
    sens, spec = sens_spec(ys, ps, threshold=0.5)
    return {"auroc": auroc, "sens": sens, "spec": spec, "y": ys, "p": ps}

def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total = 0.0
    pbar = tqdm(loader, leave=False)
    for imgs, labels in pbar:
        imgs = imgs.to(device); labels = labels.to(device)
        optimizer.zero_grad()
        logits = model(imgs).squeeze(1)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        total += loss.item()
        pbar.set_description(f"loss {loss.item():.3f}")
    return total / len(loader)

def train(model, train_loader, val_loader, optimizer, epochs, device, tag="model"):
    criterion = nn.BCEWithLogitsLoss()
    best = {"auroc": -1.0, "state": None, "epoch": 0}
    history = []
    for epoch in range(1, epochs + 1):
        t0 = time.time()
        tr_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)
        val = evaluate(model, val_loader, device)
        dt = time.time() - t0
        score = datathon_score(val["auroc"], val["sens"], val["spec"])
        history.append({"epoch": epoch, "loss": tr_loss, **{k: val[k] for k in ("auroc", "sens", "spec")}, "score": score})
        star = ""
        if val["auroc"] > best["auroc"]:
            best = {"auroc": val["auroc"], "state": copy.deepcopy(model.state_dict()), "epoch": epoch}
            star = " ★"
            torch.save(model.state_dict(), CKPT_DIR / f"{tag}_best.pt")
        print(
            f"[{tag}] ep {epoch:02d}/{epochs}  loss {tr_loss:.4f}  "
            f"AUROC {val['auroc']:.4f}  sens {val['sens']:.3f}  spec {val['spec']:.3f}  "
            f"score {score:.4f}  ({dt:.1f}s){star}"
        )
    model.load_state_dict(best["state"])
    return model, history, best

---
## Stage 1: Frozen ResNet18 backbone

All pretrained layers frozen, only a new binary head (`512 → 1`) is trained. ResNet18's `model.fc` is the final Linear layer we replace.

In [ ]:
def build_frozen():
    model = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
    for p in model.parameters():
        p.requires_grad = False
    # ResNet18 final feature size = 512
    model.fc = nn.Sequential(nn.Dropout(0.3), nn.Linear(512, 1))
    return model

model_frozen = build_frozen().to(device)
n_trainable = sum(p.numel() for p in model_frozen.parameters() if p.requires_grad)
n_total = sum(p.numel() for p in model_frozen.parameters())
print(f"Trainable: {n_trainable:,} / {n_total:,} ({100*n_trainable/n_total:.2f}%)")

opt_frozen = optim.Adam(model_frozen.fc.parameters(), lr=1e-3)
model_frozen, hist_frozen, best_frozen = train(
    model_frozen, train_loader, val_loader, opt_frozen, epochs=5, device=device, tag="resnet18_stage1_frozen"
)

---
## Stage 2: Fine-tune `layer4`

Unfreeze the final residual block group (`layer4`) with a tiny LR (1e-5), head with a bigger LR (1e-4). In ResNet18, `layer4` has 2 basic blocks totaling ~8.4M params — most of the model's capacity.

In [ ]:
def build_finetune():
    model = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
    for p in model.parameters():
        p.requires_grad = False
    for p in model.layer4.parameters():
        p.requires_grad = True
    model.fc = nn.Sequential(nn.Dropout(0.3), nn.Linear(512, 1))
    return model

model_ft = build_finetune().to(device)
n_trainable = sum(p.numel() for p in model_ft.parameters() if p.requires_grad)
n_total = sum(p.numel() for p in model_ft.parameters())
print(f"Trainable: {n_trainable:,} / {n_total:,} ({100*n_trainable/n_total:.2f}%)")

opt_ft = optim.AdamW([
    {"params": model_ft.layer4.parameters(), "lr": 1e-5},
    {"params": model_ft.fc.parameters(), "lr": 1e-4},
])
model_ft, hist_ft, best_ft = train(
    model_ft, train_loader, val_loader, opt_ft, epochs=5, device=device, tag="resnet18_stage2_finetune"
)

---
## Stage 3: Fine-tune + augmentation

Same architecture as Stage 2, with mild augmentation during training.

In [ ]:
train_tf_aug = build_transform(IMG_SIZE, augment=True)
train_ds_aug = CardiomegalyDataset(train_df, TRAIN_IMG_DIR, transform=train_tf_aug)
train_loader_aug = DataLoader(train_ds_aug, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)

model_aug = build_finetune().to(device)
opt_aug = optim.AdamW([
    {"params": model_aug.layer4.parameters(), "lr": 1e-5},
    {"params": model_aug.fc.parameters(), "lr": 1e-4},
])
model_aug, hist_aug, best_aug = train(
    model_aug, train_loader_aug, val_loader, opt_aug, epochs=10, device=device, tag="resnet18_stage3_aug"
)

---
## Compare all three stages

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
for tag, model in [("frozen", model_frozen), ("finetune", model_ft), ("aug", model_aug)]:
    val = evaluate(model, val_loader, device)
    fpr, tpr, _ = roc_curve(val["y"], val["p"])
    ax.plot(fpr, tpr, lw=2, label=f"{tag}  AUROC={val['auroc']:.3f}")
ax.plot([0, 1], [0, 1], "k--", lw=1, label="random")
ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
ax.set_title("Validation ROC — ResNet18 three stages")
ax.legend(loc="lower right"); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

---
## Pick best model + best threshold, then score

In [ ]:
candidates = {
    "stage1_frozen": model_frozen,
    "stage2_finetune": model_ft,
    "stage3_aug": model_aug,
}

summary = []
for name, m in candidates.items():
    v = evaluate(m, val_loader, device)
    thr = best_threshold(v["y"], v["p"])
    sens_t, spec_t = sens_spec(v["y"], v["p"], threshold=thr)
    score = datathon_score(v["auroc"], sens_t, spec_t)
    summary.append({"model": name, "auroc": v["auroc"], "thr": thr, "sens": sens_t, "spec": spec_t, "score": score})

summary_df = pd.DataFrame(summary).sort_values("score", ascending=False)
print(summary_df.to_string(index=False))

best_name = summary_df.iloc[0]["model"]
best_thr = float(summary_df.iloc[0]["thr"])
best_model = candidates[best_name]
print(f"\nBest: {best_name}  thr={best_thr:.3f}")

---
## Generate submission CSV for the 176 test images

In [ ]:
test_files = sorted(p.name for p in TEST_IMG_DIR.iterdir() if p.suffix.lower() == ".png")
print(f"Test images: {len(test_files)}")

test_df = pd.DataFrame({"image_file": test_files, "label": 0})
test_tf = build_transform(IMG_SIZE, augment=False)
test_ds = CardiomegalyDataset(test_df, TEST_IMG_DIR, transform=test_tf, return_label=False)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

best_model.eval()
all_probs, all_names = [], []
with torch.no_grad():
    for imgs, names in tqdm(test_loader, desc="test inference"):
        imgs = imgs.to(device)
        logits = best_model(imgs).squeeze(1)
        logits_flip = best_model(torch.flip(imgs, dims=[3])).squeeze(1)
        probs = (torch.sigmoid(logits) + torch.sigmoid(logits_flip)) / 2
        all_probs.extend(probs.cpu().numpy().tolist())
        all_names.extend(names)

sub = pd.DataFrame({"image_file": all_names, "prob": all_probs})
sub["pred"] = (sub["prob"] >= best_thr).astype(int)
sub = sub.sort_values("image_file").reset_index(drop=True)

stamp = time.strftime("%Y%m%d_%H%M")
out_path = PRED_DIR / f"submission_resnet18_{best_name}_{stamp}.csv"
sub.to_csv(out_path, index=False)
print(f"\nWrote {out_path}")
print(sub.head())
print(f"\nPositive rate in submission: {sub['pred'].mean():.3f}")